# Template File for EDA Profiler

This notebook is a template file for the running of the Chisel geospatial image dataset EDA profiler. 

Duplicate this file and edit it to suit your needs.

In [ ]:
# --- Imports --- #
import os

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from pipeline import process_dataset_incremental
from libs.utils import PipelineUtils
from libs.file_handler import FileHandler
from models.metadata import SpatialMetadata
from eda import DatasetEDA
from config import config, CLASS_ID_MAP


# Step 1: Set-up

## Args for EDA Class:

|Arg | Type | Description |
|:-|:-:|:-|
|**data_dir**| String | Path to the directory of the dataset you want to profile.|
|**set_name**| String | Name you want to assign to the dataset you are studying. Try to pick something short and logical.|
|**class_id**| Integer | 1 for building, 2 for vegetation, 3 for water, 4 for road|
|**class_name**| String | 'building', 'vegetation', 'water', 'road'. The config file will fill in this field for you.|

In [ ]:
# --- Set up parameters --- #
data_dir = '../../../data/OpenEarthMap_final' # directory containing the dataset
set_name = 'openearthmap-full' # choose an appropriate name for the dataset 
class_id = 1  # put 1 for 'building', 2 for 'vegetation', 3 for 'water', 4 for 'road'
class_name = CLASS_ID_MAP[class_id] 

# Step 1b: Inputting Spatial Metadata Manually

Some datasets present the data as image format files without any spatial metadata, but provide some information in the data manifest. In this scenario, uncomment the block below and fill in the relevant information. If you do not have the information, leave the field as None.

## Args for Spatial Metadata

|Arg | Type | Description |
|:-|:-:|:-|
|**source_path**| String | Location of original image. The file will fill this in for you. |
|**epsg**| Int | European Petroleum Survey Group reference numbers for coordinate reference systems. Usually 4-5 digits long. |
|**crs_wkt**| String | Coordinate Reference System in Well-Known-Text format. |
|**projection_format**| String | Projection format. The most common is Transverse Mercator. |
|**bounds**| Tuple[float, float, float, float] | Bottom left and top right coordinates. |
|**transform**| Tuple[float, float, float, float, float, float] | GT(0) x-coordinate of the upper-left corner of the upper-left pixel. <br> GT(1) w-e pixel resolution / pixel width. <br> GT(2) row rotation (typically zero).<br> GT(3) y-coordinate of the upper-left corner of the upper-left pixel.<br> GT(4) column rotation (typically zero).<br> GT(5) n-s pixel resolution / pixel height (negative value for a north-up image).|
|**x_cm_per_pixel**| float | x cm per pixel of the image.|
|**y_cm_per_pixel**| float | y cm per pixel of the image.|
|**mean_cm_per_pixel**| float | mean cm per pixel of the image.|
|**num_bands**| Int | Number of bands in the image|
|**centroid_crs**| Tuple[float, float] | Centroid of the image in the source CRS. |
|**centroid_wgs84**| Tuple[float, float] | Centroid of the image in WGS84 coordinates. |
|**country**| String | Country that the image is taken of. Please capitalise the first letter. |

In [ ]:
spatial_metadata = SpatialMetadata(
    source_path=data_dir,
    epsg=None,
    crs_wkt=None,
    projection_format=None,
    bounds=None,
    transform=None,
    x_cm_per_pixel=None,
    y_cm_per_pixel=None,
    mean_cm_per_pixel=None,
    num_bands=None,
    centroid_src=None,
    centroid_wgs84=None,
    country=None
)

# Step 2: Running the profiler

## Args for EDA Report:


|Arg | Type | Description |
|:-|:-:|:-|
|**stage**| String | 'pre' for pre-processing, 'post' for post-processing. Used for labeling/reporting.|
|**show_inline**| Boolean | Whether to show plots in the notebook itself.|
|**save_pdf**| Boolean | Whether to save the PDF report.|
|**save_images**| Boolean | Whether to save the images.|
|**save_summary**| Boolean | Whether to save the summary JSON file.|
|**output_dir**| Boolean | The directory to save the report and artefacts.|

**NOTE**: If your dataset is image based (no spatial metadata), please open step 1b above and fill in what you can. Then change the input below to 
```
eda_pre = DatasetEDA(data_dir, class_id, set_name, config, stage='pre', spatial_metadata=spatial_metadata)
``` 
instead of `spatial_metadata=None`


In [ ]:
# --- EDA (Pre-processing) --- #
eda_pre = DatasetEDA(data_dir, class_id, set_name, config, stage='pre', spatial_metadata=None) 
eda_pre.report(show_inline=False, save_pdf=True, save_images=True, save_summary=True, output_dir='reports')


# Step 3: Process the Dataset

In [ ]:
# --- Process the dataset --- #
process_dataset_incremental(
    data_dir=data_dir,
    set_name=set_name,
    class_id=class_id,
    config=config
)

In [7]:
# --- Clean the dataset --- #
quarantine_dir = config['quarantine_dir']
set_dir = os.path.join(quarantine_dir, set_name)
utils = PipelineUtils(config, set_dir)
utils.clean_dataset(f'{class_name}') 

Cleaning YOLO label files for class 'building'...
Removing unlabelled images and updating CSV for class 'building'...
Updating CSV deletion status for class 'building'...
Dataset cleaning complete.


# Step 4: Run EDA Profiler on Processed Dataset

In [ ]:
# --- EDA (Post-processing) --- #
# The output directory and class name must match your class_id above 
class_dir = os.path.join(set_dir, class_name)
eda_post = DatasetEDA(class_dir, class_id, set_name, config, stage='post', spatial_metadata=None)
eda_post.report(show_inline=False, save_pdf=True, save_images=True, save_summary=True, output_dir='reports')

# Step 5: Moving From Quarantine to Main

Set `all_sets=True` if you want to move all the sets in quarantine (in the class_id you set earlier) to main. 

Set `del_original=True` if you want to delete the quarantine files after merging.

In [ ]:
# --- Move the quarantined files to main dataset --- #
# NOTE: only do this step if you are certain this is a good dataset
file_handler = FileHandler(config)
file_handler.merge_quarantine_sets(set_dir, all_sets=False, del_original=True) 